# Universal PubMed Scraping Notebook

This notebook is used to collect biomedical article metadata and abstracts from PubMed using the NCBI Entrez API.

The notebook is designed as a reusable scraping pipeline. Instead of creating separate notebooks for different year ranges, the scraping period, target number of articles, and output file names are controlled through a configuration section.

The scraped data will be used as the raw corpus for a biomedical NLP final project. The downstream pipeline includes exploratory data analysis, preprocessing, distant supervision, biomedical named entity recognition, knowledge graph construction, and drug repurposing analysis.

Main objectives of this notebook:

1. Retrieve PubMed article IDs based on a biomedical query and publication year range.
2. Fetch article metadata, titles, abstracts, journals, and publication years.
3. Validate and clean the collected records.
4. Save the scraped data into CSV files for downstream EDA and preprocessing.

## Import Libraries

This section imports the required Python libraries for accessing the PubMed API, handling tabular data, parsing article metadata, controlling execution speed, and saving the scraping results.

The notebook uses `Bio.Entrez` from Biopython to communicate with the NCBI PubMed API.

In [1]:
import time
import pandas as pd
from Bio import Entrez
from tqdm.auto import tqdm

d:\AI_Journey\NLP\nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Scraping Configuration

This section defines the main configuration variables for the scraping process.

The year range, target number of articles, minimum abstract length, and output file names are controlled from this cell. This makes the notebook reusable for different PubMed scraping periods without duplicating the entire notebook.

To scrape a different period, only change `MIN_YEAR` and `MAX_YEAR`.

In [ ]:
# NCBI requires users to provide an email when using Entrez
# Replace this email before running the notebook with the NCBI Entrez API
Entrez.email = "your_email@example.com"

# Main scraping configuration
TARGET_COUNT = 2000
MIN_YEAR = 2022
MAX_YEAR = 2026
MIN_ABSTRACT_WORDS = 10

# Automatically generated label for file naming
YEAR_RANGE_LABEL = f"{MIN_YEAR}_{MAX_YEAR}"

# Output file names
RAW_OUTPUT_PATH = f"pubmed_abstracts_{YEAR_RANGE_LABEL}_{TARGET_COUNT}_raw.csv"
CLEAN_OUTPUT_PATH = f"pubmed_abstracts_{YEAR_RANGE_LABEL}_clean.csv"
TEXT_OUTPUT_PATH = f"pubmed_abstracts_{YEAR_RANGE_LABEL}_clean_with_text.csv"

print("Scraping configuration")
print("Target count:", TARGET_COUNT)
print("Year range:", MIN_YEAR, "-", MAX_YEAR)
print("Minimum abstract words:", MIN_ABSTRACT_WORDS)
print("Raw output path:", RAW_OUTPUT_PATH)
print("Clean output path:", CLEAN_OUTPUT_PATH)
print("Text output path:", TEXT_OUTPUT_PATH)

Scraping configuration
Target count: 2000
Year range: 2022 - 2026
Minimum abstract words: 10
Raw output path: pubmed_abstracts_2022_2026_2000_raw.csv
Clean output path: pubmed_abstracts_2022_2026_clean.csv
Text output path: pubmed_abstracts_2022_2026_clean_with_text.csv


## PubMed Query Definition

This section defines the biomedical search query used to retrieve PubMed articles.

The query focuses on biomedical topics related to viral infection, host response, genes, diseases, chemicals, and therapeutic or drug-related terms. These topics are aligned with the downstream project goal: building a biomedical NER dataset and using the extracted entities for knowledge graph construction and drug repurposing analysis.

The publication year filter is handled separately through the Entrez search parameters, so the query itself only describes the biomedical topic scope.

In [3]:
PUBMED_QUERY = """
(
    "viral infection"[Title/Abstract]
    OR "virus"[Title/Abstract]
    OR "SARS-CoV-2"[Title/Abstract]
    OR "COVID-19"[Title/Abstract]
    OR "influenza"[Title/Abstract]
    OR "HIV"[Title/Abstract]
    OR "hepatitis"[Title/Abstract]
    OR "dengue"[Title/Abstract]
)
AND
(
    "gene"[Title/Abstract]
    OR "protein"[Title/Abstract]
    OR "host response"[Title/Abstract]
    OR "immune response"[Title/Abstract]
    OR "disease"[Title/Abstract]
    OR "drug"[Title/Abstract]
    OR "therapy"[Title/Abstract]
    OR "treatment"[Title/Abstract]
)
"""

print(PUBMED_QUERY)


(
    "viral infection"[Title/Abstract]
    OR "virus"[Title/Abstract]
    OR "SARS-CoV-2"[Title/Abstract]
    OR "COVID-19"[Title/Abstract]
    OR "influenza"[Title/Abstract]
    OR "HIV"[Title/Abstract]
    OR "hepatitis"[Title/Abstract]
    OR "dengue"[Title/Abstract]
)
AND
(
    "gene"[Title/Abstract]
    OR "protein"[Title/Abstract]
    OR "host response"[Title/Abstract]
    OR "immune response"[Title/Abstract]
    OR "disease"[Title/Abstract]
    OR "drug"[Title/Abstract]
    OR "therapy"[Title/Abstract]
    OR "treatment"[Title/Abstract]
)



## Retrieve PubMed IDs

This section defines a helper function to retrieve PubMed article IDs using the Entrez `esearch` endpoint.

The function uses the configured biomedical query, publication year range, and target article count. The year range is passed through `mindate` and `maxdate`, making the scraping period controlled by the configuration cell.

The output of this function is a list of PubMed IDs. These IDs will be used in the next step to fetch full article metadata and abstracts.

In [4]:
def retrieve_pubmed_ids(query, target_count, min_year, max_year):
    # This function retrieves PubMed IDs based on the search query and year range
    handle = Entrez.esearch(
        db="pubmed",
        term=query,
        retmax=target_count,
        sort="relevance",
        datetype="pdat",
        mindate=f"{min_year}/01/01",
        maxdate=f"{max_year}/12/31"
    )
    
    search_results = Entrez.read(handle)
    handle.close()
    
    pubmed_ids = search_results["IdList"]
    
    return pubmed_ids

## Test PubMed ID Retrieval

This cell runs the PubMed ID retrieval function using the current configuration.

For repository documentation, this notebook can be left without cell outputs. If the notebook needs to be tested, reduce `TARGET_COUNT` to a small number first to avoid long API calls.

In [5]:
pubmed_ids = retrieve_pubmed_ids(
    query=PUBMED_QUERY,
    target_count=TARGET_COUNT,
    min_year=MIN_YEAR,
    max_year=MAX_YEAR
)

print("Retrieved PubMed IDs:", len(pubmed_ids))
print(pubmed_ids[:10])

Retrieved PubMed IDs: 2000
['39732271', '38892370', '37243166', '35216673', '39961996', '39353439', '41160032', '36458986', '36505482', '37704050']


## Fetch Article Metadata and Abstracts

This section defines helper functions to fetch and parse PubMed article metadata from a list of PubMed IDs.

The metadata extracted from each article includes:

1. PubMed ID
2. Article title
3. Abstract text
4. Journal name
5. Publication year
6. PubMed URL

Only articles with valid titles, abstracts, and publication years will be kept. This filtering is important because downstream NLP tasks require complete text data.

In [6]:
def extract_publication_year(article):
    # This function extracts the publication year from a PubMed article record
    try:
        pub_date = article["MedlineCitation"]["Article"]["Journal"]["JournalIssue"]["PubDate"]
        
        if "Year" in pub_date:
            return int(pub_date["Year"])
        
        if "MedlineDate" in pub_date:
            medline_date = str(pub_date["MedlineDate"])
            year_candidate = medline_date[:4]
            
            if year_candidate.isdigit():
                return int(year_candidate)
    
    except Exception:
        return None
    
    return None


def extract_abstract_text(article):
    # This function extracts and joins abstract sections from a PubMed article record
    try:
        abstract_parts = article["MedlineCitation"]["Article"]["Abstract"]["AbstractText"]
        abstract_text = " ".join(str(part) for part in abstract_parts)
        return abstract_text.strip()
    
    except Exception:
        return None


def extract_article_record(article):
    # This function converts a raw PubMed article record into a clean dictionary
    try:
        medline = article["MedlineCitation"]
        article_data = medline["Article"]
        
        pmid = str(medline["PMID"])
        title = str(article_data.get("ArticleTitle", "")).strip()
        abstract = extract_abstract_text(article)
        journal = str(article_data.get("Journal", {}).get("Title", "")).strip()
        publication_year = extract_publication_year(article)
        url = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
        
        if not title or not abstract or publication_year is None:
            return None
        
        record = {
            "pmid": pmid,
            "title": title,
            "abstract": abstract,
            "journal": journal,
            "publication_year": publication_year,
            "url": url
        }
        
        return record
    
    except Exception:
        return None

## Fetch PubMed Articles in Batches

This section defines a function to fetch PubMed article details in batches using the Entrez `efetch` endpoint.

Batching is used to avoid requesting all articles at once. This makes the scraping process more stable and more respectful of the PubMed API. A short delay is also added between requests to reduce the risk of hitting API rate limits.

In [7]:
def fetch_pubmed_records(pubmed_ids, batch_size=100, sleep_time=0.4):
    # This function fetches PubMed article records in batches
    records = []
    
    for start_idx in tqdm(range(0, len(pubmed_ids), batch_size)):
        batch_ids = pubmed_ids[start_idx:start_idx + batch_size]
        
        handle = Entrez.efetch(
            db="pubmed",
            id=",".join(batch_ids),
            rettype="medline",
            retmode="xml"
        )
        
        fetch_results = Entrez.read(handle)
        handle.close()
        
        articles = fetch_results["PubmedArticle"]
        
        for article in articles:
            record = extract_article_record(article)
            
            if record is not None:
                records.append(record)
        
        time.sleep(sleep_time)
    
    return records

## Run PubMed Scraping

This section runs the scraping process using the PubMed IDs retrieved earlier.

The scraping function fetches article metadata and abstracts from PubMed in batches. The result is a list of article records, where each record contains the title, abstract, journal, publication year, PMID, and PubMed URL.

For repository documentation, this notebook can be left without output. To test the notebook quickly, use a smaller `TARGET_COUNT` before running this cell.

In [8]:
raw_records = fetch_pubmed_records(
    pubmed_ids=pubmed_ids,
    batch_size=100,
    sleep_time=0.4
)

print("Fetched valid records:", len(raw_records))

100%|██████████| 20/20 [01:31<00:00,  4.57s/it]

Fetched valid records: 1971


## Convert Scraped Records to DataFrame

This section converts the scraped PubMed records into a pandas DataFrame.

A tabular format makes the data easier to inspect, validate, clean, and save. Each row represents one PubMed article, while each column stores one metadata field or text field.

In [9]:
pubmed_df = pd.DataFrame(raw_records)

print("DataFrame shape:", pubmed_df.shape)
pubmed_df.head()

DataFrame shape: (1971, 6)


,pmid,title,abstract,journal,publication_year,url
0,39732271,"Dengue virus: Etiology, epidemiology, pathobio...",Dengue flavivirus (DENV) is the virus that cau...,"Infection, genetics and evolution : journal of...",2025,https://pubmed.ncbi.nlm.nih.gov/39732271/
1,38892370,Immune Response to Respiratory Viral Infections.,The respiratory system is constantly exposed t...,International journal of molecular sciences,2024,https://pubmed.ncbi.nlm.nih.gov/38892370/
2,37243166,Treatment Options for Hepatitis A and E: A Non...,Hepatitis A and hepatitis E are relatively com...,Viruses,2023,https://pubmed.ncbi.nlm.nih.gov/37243166/
3,35216673,A blood atlas of COVID-19 defines hallmarks of...,Treatment of severe COVID-19 is currently limi...,Cell,2022,https://pubmed.ncbi.nlm.nih.gov/35216673/
4,39961996,Viral sepsis - pathophysiology and disease man...,Viral infection is found in approximately 30% ...,Infection,2025,https://pubmed.ncbi.nlm.nih.gov/39961996/


## Validate Publication Year Range

This section validates whether all collected articles fall within the configured publication year range.

Although the Entrez search already applies a publication date filter, this additional validation step is useful as a safety check. Any records outside the configured range will be removed before saving the dataset.

In [10]:
pubmed_df = pubmed_df[
    pubmed_df["publication_year"].between(MIN_YEAR, MAX_YEAR)
].copy()

print("DataFrame shape after year validation:", pubmed_df.shape)
print(pubmed_df["publication_year"].value_counts().sort_index())

DataFrame shape after year validation: (1962, 6)
publication_year
2022    718
2023    461
2024    365
2025    290
2026    128
Name: count, dtype: int64


## Basic Data Inspection

This section performs a basic inspection of the scraped PubMed dataset.

The goal is to understand the structure of the collected data, check available columns, inspect missing values, and review the distribution of publication years. This step helps ensure that the scraped dataset is suitable for downstream preprocessing.

In [11]:
print("Dataset shape:", pubmed_df.shape)
print("\nColumns:")
print(pubmed_df.columns.tolist())

print("\nMissing values:")
print(pubmed_df.isna().sum())

print("\nPublication year distribution:")
print(pubmed_df["publication_year"].value_counts().sort_index())

Dataset shape: (1962, 6)

Columns:
['pmid', 'title', 'abstract', 'journal', 'publication_year', 'url']

Missing values:
pmid                0
title               0
abstract            0
journal             0
publication_year    0
url                 0
dtype: int64

Publication year distribution:
publication_year
2022    718
2023    461
2024    365
2025    290
2026    128
Name: count, dtype: int64


## Check Duplicate Articles

This section checks duplicate PubMed articles using the `pmid` column.

Since PMID is a unique identifier for PubMed articles, duplicate PMID values indicate duplicated records in the scraped dataset. Removing duplicates is important to avoid repeated documents in downstream EDA, preprocessing, model training, and knowledge graph construction.

In [12]:
duplicate_pmid_count = pubmed_df.duplicated(subset=["pmid"]).sum()

print("Duplicate PMID count:", duplicate_pmid_count)

pubmed_df = pubmed_df.drop_duplicates(subset=["pmid"]).copy()

print("Dataset shape after duplicate removal:", pubmed_df.shape)

Duplicate PMID count: 0
Dataset shape after duplicate removal: (1962, 6)


## Check Abstract Length

This section calculates the number of words in each abstract.

Very short abstracts may not contain enough biomedical context for downstream named entity recognition and knowledge graph construction. Therefore, abstract length is used as one of the quality indicators before saving the cleaned dataset.

In [13]:
pubmed_df["abstract_word_count"] = pubmed_df["abstract"].astype(str).str.split().str.len()

print("Abstract word count summary:")
print(pubmed_df["abstract_word_count"].describe())

print("\nShortest abstracts:")
pubmed_df[["pmid", "publication_year", "title", "abstract_word_count"]].sort_values(
    by="abstract_word_count"
).head(10)

Abstract word count summary:
count    1962.000000
mean      198.282365
std        65.396292
min        17.000000
25%       152.000000
50%       194.000000
75%       236.000000
max       723.000000
Name: abstract_word_count, dtype: float64

Shortest abstracts:


,pmid,publication_year,title,abstract_word_count
21,34531275,2022,Convalescent plasma for SARS-CoV-2 infection: ...,17
595,36042769,2022,Not the Virus but Treatment and Immune Respons...,24
1428,38543884,2024,Current Understanding of the Immune Response a...,27
461,33649117,2023,The anti-influenza virus drug favipiravir has ...,34
1482,38012410,2023,Severe dengue progression beyond enhancement.,35
1372,39176101,2024,Cholestatic hepatitis in acute Epstein-Barr vi...,37
1461,37240555,2023,Long COVID Syndrome: Lesson Learned and Future...,37
935,39610406,2024,Perceptions and Barriers to Outpatient Antivir...,40
453,36410387,2022,Monoclonal Antibodies as Long-Acting Products:...,40
1289,36299425,2022,High co-circulation of influenza and SARS-CoV-2.,43


## Filter Short Abstracts

This section removes articles with abstracts shorter than the configured minimum word count.

The default threshold is controlled by `MIN_ABSTRACT_WORDS` in the configuration section. This filtering step helps remove records with incomplete or uninformative abstracts.

In [14]:
pubmed_df_clean = pubmed_df[
    pubmed_df["abstract_word_count"] >= MIN_ABSTRACT_WORDS
].copy()

print("Dataset shape before short abstract filtering:", pubmed_df.shape)
print("Dataset shape after short abstract filtering:", pubmed_df_clean.shape)

Dataset shape before short abstract filtering: (1962, 7)
Dataset shape after short abstract filtering: (1962, 7)


## Create Combined Text Column

This section creates a combined text column by joining the article title and abstract.

The downstream NLP pipeline works on biomedical text, and combining the title with the abstract provides richer context than using the abstract alone. This format is also aligned with common biomedical NER datasets such as BC5CDR, which are based on PubMed titles and abstracts.

In [15]:
pubmed_df_clean["text"] = (
    pubmed_df_clean["title"].astype(str).str.strip()
    + " "
    + pubmed_df_clean["abstract"].astype(str).str.strip()
)

pubmed_df_clean["text_word_count"] = pubmed_df_clean["text"].str.split().str.len()

print("Text word count summary:")
print(pubmed_df_clean["text_word_count"].describe())

pubmed_df_clean[["pmid", "publication_year", "title", "abstract_word_count", "text_word_count"]].head()

Text word count summary:
count    1962.000000
mean      211.937819
std        66.671229
min        25.000000
25%       166.000000
50%       208.000000
75%       252.000000
max       732.000000
Name: text_word_count, dtype: float64


,pmid,publication_year,title,abstract_word_count,text_word_count
0,39732271,2025,"Dengue virus: Etiology, epidemiology, pathobio...",149,164
1,38892370,2024,Immune Response to Respiratory Viral Infections.,104,110
2,37243166,2023,Treatment Options for Hepatitis A and E: A Non...,299,309
3,35216673,2022,A blood atlas of COVID-19 defines hallmarks of...,150,162
4,39961996,2025,Viral sepsis - pathophysiology and disease man...,256,263


## Save Scraped Dataset

This section saves the scraped PubMed dataset into CSV files.

Three output files are generated:

1. Raw scraped data after basic year validation.
2. Clean data after duplicate removal and short abstract filtering.
3. Clean data with a combined `text` column.

The final `text` dataset will be used in the next notebook for sentence-level preprocessing and tokenization.

In [ ]:
pubmed_df.to_csv(RAW_OUTPUT_PATH, index=False)
pubmed_df_clean.to_csv(CLEAN_OUTPUT_PATH, index=False)
pubmed_df_clean.to_csv(TEXT_OUTPUT_PATH, index=False)

print("Saved files:")
print("Raw dataset:", RAW_OUTPUT_PATH)
print("Clean dataset:", CLEAN_OUTPUT_PATH)
print("Text dataset:", TEXT_OUTPUT_PATH)

## Final Scraping Summary

This section summarizes the result of the PubMed scraping pipeline.

The cleaned dataset contains PubMed articles within the configured year range, with valid titles, abstracts, publication years, and PubMed URLs. The dataset also includes a combined `text` column created from the article title and abstract.

This scraped corpus will be used as the unlabeled biomedical text source for the next stages of the final project:

1. Exploratory data analysis
2. Sentence-level preprocessing
3. Distant supervision using biomedical dictionaries
4. Biomedical NER model training
5. Knowledge graph construction
6. Drug repurposing analysis

In [16]:
summary = {
    "target_count": TARGET_COUNT,
    "min_year": MIN_YEAR,
    "max_year": MAX_YEAR,
    "retrieved_pubmed_ids": len(pubmed_ids),
    "raw_valid_records": len(pubmed_df),
    "clean_records": len(pubmed_df_clean),
    "min_abstract_words": MIN_ABSTRACT_WORDS,
    "raw_output_path": RAW_OUTPUT_PATH,
    "clean_output_path": CLEAN_OUTPUT_PATH,
    "text_output_path": TEXT_OUTPUT_PATH
}

summary

{'target_count': 2000,
 'min_year': 2022,
 'max_year': 2026,
 'retrieved_pubmed_ids': 2000,
 'raw_valid_records': 1962,
 'clean_records': 1962,
 'min_abstract_words': 10,
 'raw_output_path': 'pubmed_abstracts_2022_2026_2000_raw.csv',
 'clean_output_path': 'pubmed_abstracts_2022_2026_clean.csv',
 'text_output_path': 'pubmed_abstracts_2022_2026_clean_with_text.csv'}